# Deep SARSA on CartPole

In Lesson07 we learned SARSA with a **Q-table**: one entry $q(s, a)$ for every state-action pair.
CartPole has a **continuous** state (cart position, cart velocity, pole angle, pole angular velocity), so a table is no longer possible.
We replace the table by a neural network $\hat q(s, a; w)$ that receives the state $s$ and returns one value per action.

**Sarsa with function approximation** (Zhao, *Mathematical Foundations of RL*, Sec. 8.3.1, Algorithm 8.2). After every transition $(s_t, a_t, r_{t+1}, s_{t+1}, a_{t+1})$ the weights are updated as

$$
w_{t+1} = w_t + \alpha_t \Big[ r_{t+1} + \gamma\, \hat q(s_{t+1}, a_{t+1}; w_t) - \hat q(s_t, a_t; w_t) \Big] \nabla_w \hat q(s_t, a_t; w_t),
$$

and the policy is improved $\varepsilon$-greedily with respect to $\hat q$:

$$
\pi_{t+1}(a \mid s_t) =
\begin{cases}
1 - \dfrac{\varepsilon}{|\mathcal A(s_t)|}\big(|\mathcal A(s_t)| - 1\big), & a = \arg\max_{a \in \mathcal A(s_t)} \hat q(s_t, a; w_{t+1}), \\[2mm]
\dfrac{\varepsilon}{|\mathcal A(s_t)|}, & \text{otherwise.}
\end{cases}
$$

Remarks:

* It is a **semi-gradient** method: the target $r_{t+1} + \gamma \hat q(s_{t+1}, a_{t+1}; w_t)$ also depends on $w$, but we treat it as a constant and only differentiate $\hat q(s_t, a_t; w_t)$.
* It is **on-policy**: $a_{t+1}$ is chosen by the *same* $\varepsilon$-greedy policy that generates the data. Q-learning (and DQN) would use $\max_a \hat q(s_{t+1}, a; w_t)$ instead.

In [ ]:
import copy
import random

import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
from matplotlib import pyplot as plt

## Configuration

In [ ]:
NUM_EPISODES = 900
MAX_STEPS_PER_EPISODE = 500
GAMMA = 0.99
LEARNING_RATE = 0.0005
HIDDEN_SIZE = 128
EPSILON_MAX = 1.0
EPSILON_MIN = 0.01
EPSILON_DECAY = 0.01
EVAL_EVERY = 10         # check the greedy policy every EVAL_EVERY episodes
EVAL_EPISODES = 10      # number of greedy episodes used in each check
SEED = 42

## Environment

* State $s \in \mathbb{R}^4$: cart position, cart velocity, pole angle, pole angular velocity.
* Actions: `0` push the cart to the left, `1` push the cart to the right.
* Reward $+1$ for every step. The episode **terminates** when the pole angle exceeds $\pm 12^\circ$ or the cart position exceeds $\pm 2.4$, and it is **truncated** after 500 steps.
* CartPole-v1 is considered solved when the average return is at least 475.

In [ ]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

env = gym.make("CartPole-v1", max_episode_steps=MAX_STEPS_PER_EPISODE)
env.action_space.seed(SEED)

state_size = env.observation_space.shape[0]
action_size = env.action_space.n

print("State size :", state_size)
print("Action size:", action_size)

## Q-network

The network plays the role of the Q-table: input $s$ (4 numbers), output $[\hat q(s, \text{left}; w),\ \hat q(s, \text{right}; w)]$.

In [ ]:
class QNetwork(nn.Module):
    """Map a CartPole observation to one Q-value per action."""

    def __init__(self, state_size, action_size, hidden_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, action_size),
        )

    def forward(self, state):
        return self.net(state)

In [ ]:
q_net = QNetwork(state_size, action_size, HIDDEN_SIZE)
optimizer = torch.optim.Adam(q_net.parameters(), lr=LEARNING_RATE)
loss_fn = nn.MSELoss()

print(q_net)
print("Number of parameters:", sum(p.numel() for p in q_net.parameters()))

## Policy helper

Same $\varepsilon$-greedy rule as in the tabular case, but $\arg\max_a$ is taken over the network output.

In [ ]:
def epsilon_greedy(state, epsilon):
    """Choose a random action or the action with the largest Q-value."""
    if random.random() < epsilon:
        return env.action_space.sample()
    state_tensor = torch.as_tensor(state, dtype=torch.float32).unsqueeze(0)
    with torch.no_grad():
        q_values = q_net(state_tensor)
    return int(q_values.argmax(dim=1).item())

## Learning step

Instead of writing the gradient update by hand, we let PyTorch do it. For one transition we minimise

$$
J(w) = \big( y_t - \hat q(s_t, a_t; w) \big)^2,
\qquad
y_t =
\begin{cases}
r_{t+1}, & \text{if } s_{t+1} \text{ is terminal}, \\
r_{t+1} + \gamma\, \hat q(s_{t+1}, a_{t+1}; w_t), & \text{otherwise,}
\end{cases}
$$

where $y_t$ is computed with `torch.no_grad()` (treated as a constant). Then

$$
-\tfrac{1}{2}\nabla_w J(w) = \big( y_t - \hat q(s_t, a_t; w) \big) \nabla_w \hat q(s_t, a_t; w),
$$

so one gradient-descent step on $J$ is exactly the SARSA update of the book. (`Adam` only adapts the step size $\alpha_t$ for every weight.)

Note: we stop bootstrapping only when the episode **terminates** (the pole fell). When it is **truncated** at 500 steps the state $s_{t+1}$ is not terminal, so we keep $\gamma\, \hat q(s_{t+1}, a_{t+1}; w_t)$.

In [ ]:
def sarsa_update(state, action, reward, next_state, next_action, terminated):
    """One semi-gradient SARSA step on the transition (s, a, r, s', a')."""
    state_tensor = torch.as_tensor(state, dtype=torch.float32).unsqueeze(0)
    next_state_tensor = torch.as_tensor(next_state, dtype=torch.float32).unsqueeze(0)

    # q_hat(s_t, a_t; w): this is the term we differentiate
    q_value = q_net(state_tensor)[0, action]

    # TD target: treated as a constant (no gradient)
    with torch.no_grad():
        next_q_value = q_net(next_state_tensor)[0, next_action]
        target = reward + GAMMA * next_q_value * (1.0 - terminated)

    loss = loss_fn(q_value, target)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return loss.item()

### Check a single update

One SARSA step should move $\hat q(s_t, a_t; w)$ towards the target $y_t$.

In [ ]:
# Work on a copy so that this check does not change the network we train below
q_net_backup = copy.deepcopy(q_net.state_dict())
optimizer_backup = copy.deepcopy(optimizer.state_dict())

state, info = env.reset(seed=SEED)
action = 0
next_state, reward, terminated, truncated, info = env.step(action)
next_action = 0

with torch.no_grad():
    q_before = q_net(torch.as_tensor(state, dtype=torch.float32))[action].item()
    target = reward + GAMMA * q_net(torch.as_tensor(next_state, dtype=torch.float32))[next_action].item()

for _ in range(10):
    sarsa_update(state, action, reward, next_state, next_action, float(terminated))

with torch.no_grad():
    q_after = q_net(torch.as_tensor(state, dtype=torch.float32))[action].item()

print(f"target y_t           : {target:.4f}")
print(f"q_hat(s,a) before    : {q_before:.4f}")
print(f"q_hat(s,a) after 10x : {q_after:.4f}")

q_net.load_state_dict(q_net_backup)
optimizer.load_state_dict(optimizer_backup)

## Greedy evaluation helper

Online deep SARSA is **unstable**: the network can learn a very good policy and forget it a few episodes later (each update also changes $\hat q$ at all the other states).
For this reason, every `EVAL_EVERY` episodes we run the greedy policy ($\varepsilon = 0$) and keep a copy of the best network found so far.

In [ ]:
def run_greedy_episode(seed):
    """Run one episode with the greedy policy and return its total reward."""
    state, info = env.reset(seed=seed)
    episode_return = 0.0
    for step in range(MAX_STEPS_PER_EPISODE):
        action = epsilon_greedy(state, epsilon=0.0)
        state, reward, terminated, truncated, info = env.step(action)
        episode_return += reward
        if terminated or truncated:
            break
    return episode_return

## Training

For every episode (Algorithm 8.2):

1. reset the environment and choose $a_0$ with the $\varepsilon$-greedy policy;
2. at every step: take $a_t$, observe $r_{t+1}, s_{t+1}$, choose $a_{t+1}$ $\varepsilon$-greedily, update $w$ with `sarsa_update`, and set $(s_t, a_t) \leftarrow (s_{t+1}, a_{t+1})$;
3. after the episode, decay $\varepsilon$: $\ \varepsilon_k = \varepsilon_{\min} + (\varepsilon_{\max} - \varepsilon_{\min})\, e^{-\lambda k}$.

The policy improvement step is implicit: `epsilon_greedy` always uses the latest $\hat q$.

In [ ]:
epsilon = EPSILON_MAX
episode_returns = []
episode_epsilons = []
eval_episodes = []
eval_returns = []

best_eval_return = -np.inf
best_weights = copy.deepcopy(q_net.state_dict())
best_episode = 0

for episode in range(NUM_EPISODES):
    state, info = env.reset(seed=SEED + episode)
    action = epsilon_greedy(state, epsilon)
    episode_return = 0.0

    for step in range(MAX_STEPS_PER_EPISODE):
        next_state, reward, terminated, truncated, info = env.step(action)
        episode_return += reward

        # SARSA: the next action comes from the same epsilon-greedy policy
        next_action = epsilon_greedy(next_state, epsilon)
        sarsa_update(state, action, reward, next_state, next_action, float(terminated))

        state = next_state
        action = next_action

        if terminated or truncated:
            break

    episode_returns.append(episode_return)
    episode_epsilons.append(epsilon)
    epsilon = EPSILON_MIN + (EPSILON_MAX - EPSILON_MIN) * np.exp(-EPSILON_DECAY * (episode + 1))

    # Check the greedy policy and keep the best network
    if (episode + 1) % EVAL_EVERY == 0:
        mean_eval = np.mean([run_greedy_episode(SEED + 5000 + i) for i in range(EVAL_EPISODES)])
        eval_episodes.append(episode + 1)
        eval_returns.append(mean_eval)
        if mean_eval > best_eval_return:
            best_eval_return = mean_eval
            best_weights = copy.deepcopy(q_net.state_dict())
            best_episode = episode + 1

    if episode == 0 or (episode + 1) % 50 == 0:
        print(
            f"Episode {episode + 1:4d} | return {episode_return:6.1f} | "
            f"avg(last 20) {np.mean(episode_returns[-20:]):6.1f} | "
            f"epsilon {episode_epsilons[-1]:.3f} | best greedy {best_eval_return:6.1f}"
        )

print(f"\nBest greedy return {best_eval_return:.1f} at episode {best_episode}")

## Learning curves

In [ ]:
window = 20
rolling_returns = np.convolve(episode_returns, np.ones(window) / window, mode="valid")

plt.figure(figsize=(10, 4))
plt.plot(episode_returns, alpha=0.3, label="training return ($\\varepsilon$-greedy)")
plt.plot(range(window - 1, NUM_EPISODES), rolling_returns, label=f"rolling average ({window} episodes)")
plt.plot(eval_episodes, eval_returns, "o-", ms=3, label="greedy policy check")
plt.axhline(475, color="red", linestyle="--", label="solved threshold (475)")
plt.axvline(best_episode, color="grey", linestyle=":", label=f"best network (episode {best_episode})")
plt.xlabel("Episode")
plt.ylabel("Return")
plt.title("Deep SARSA on CartPole-v1")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
plt.figure(figsize=(10, 3))
plt.plot(episode_epsilons)
plt.xlabel("Episode")
plt.ylabel("$\\varepsilon$")
plt.title("Exploration rate")
plt.grid(alpha=0.3)
plt.show()

## Evaluate the greedy policy

We compare the network at the **end** of training with the **best** network kept during training, using 100 new episodes.

In [ ]:
last_returns = [run_greedy_episode(SEED + 1000 + i) for i in range(100)]

q_net.load_state_dict(best_weights)
best_returns = [run_greedy_episode(SEED + 1000 + i) for i in range(100)]

for name, returns in [("last network", last_returns), ("best network", best_returns)]:
    print(
        f"{name}: mean {np.mean(returns):6.1f} | std {np.std(returns):6.1f} | "
        f"min {np.min(returns):5.0f} | max {np.max(returns):5.0f} | "
        f"solved (mean >= 475): {np.mean(returns) >= 475}"
    )

## One greedy rollout

Cart position and pole angle during one episode with the best network. The dashed lines are the limits that end the episode.

In [ ]:
state, info = env.reset(seed=SEED + 2000)
positions = [state[0]]
angles = [np.degrees(state[2])]

for step in range(MAX_STEPS_PER_EPISODE):
    action = epsilon_greedy(state, epsilon=0.0)
    state, reward, terminated, truncated, info = env.step(action)
    positions.append(state[0])
    angles.append(np.degrees(state[2]))
    if terminated or truncated:
        break

print(f"Episode length: {step + 1} steps")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(positions)
axes[0].axhline(2.4, color="red", linestyle="--")
axes[0].axhline(-2.4, color="red", linestyle="--")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Cart position")
axes[0].set_title("Cart position")
axes[0].grid(alpha=0.3)

axes[1].plot(angles)
axes[1].axhline(12, color="red", linestyle="--")
axes[1].axhline(-12, color="red", linestyle="--")
axes[1].set_xlabel("Step")
axes[1].set_ylabel("Pole angle (degrees)")
axes[1].set_title("Pole angle")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Save the trained network

To watch the agent, run from this folder:

```
python run_deep_sarsa_cartpole_policy.py --episodes 3
```

In [ ]:
MODEL_PATH = "deep_sarsa_cartpole_policy.pth"
torch.save(best_weights, MODEL_PATH)
print("Saved:", MODEL_PATH)

# Check: reload the weights in a new network
loaded_net = QNetwork(state_size, action_size, HIDDEN_SIZE)
loaded_net.load_state_dict(torch.load(MODEL_PATH, weights_only=True))
loaded_net.eval()

state, info = env.reset(seed=SEED)
state_tensor = torch.as_tensor(state, dtype=torch.float32).unsqueeze(0)
with torch.no_grad():
    print("Same greedy action:", loaded_net(state_tensor).argmax().item() == q_net(state_tensor).argmax().item())

In [ ]:
env.close()